# Figure 5, Panel D

# Setup

In [1]:
# load packages
library(tidyverse)
library(edgeR)
library(ggrastr)
library(gridExtra)
library(RColorBrewer)
library(patchwork)
library(pheatmap)
library(rtracklayer)
library(GenomicRanges)
library(here)
here::i_am("figures/Figure5/PanelD/PanelD.ipynb")
theme_set(theme_bw() + theme(panel.grid=element_blank(), text=element_text(size=15)))

hm <- function(table, rows, color, n, scale="row"){
    temp <- table[rows,]
    names <- colnames(temp)
    pheatmap(log2(temp+1), cluster_rows=FALSE, cluster_cols=FALSE, color=colorRampPalette(c("white", color))(100), scale=scale , show_rownames=FALSE, cellheight = 150/n,
             breaks=seq(-2,2,length.out=100), 
             main=paste("n=", length(rows)))
}

dex_col <- "#5791c9"
ifn_col <- "#149e77"


sessionInfo()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: limma


Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine


Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following object is masked from ‘package:gridExtra’:

    combine


The following object is masked from ‘package:limma’:

    plotMA


The following objects are masked from ‘package:lub

R version 4.2.2 Patched (2022-11-10 r83330)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 23.04

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] here_1.0.2           rtracklayer_1.58.0   GenomicRanges_1.50.2
 [4] GenomeInfoDb_1.34.9  IRanges_2.32.0       S4Vectors_0.36.2    
 [7] BiocGenerics_0.44.0  pheatmap_1.0.12      patchwork_1.3.0     
[10] RColorBrewer_1.1-3   gridExtra_2.3        ggrastr_1.0.2       
[13] edg

# Load data

In [2]:
# read in counts data
countmat_1h <- read.delim(
  here("inputs", "countmat", "GM12878_stimulus", "GM_stim_1h_counts_1.tsv"),
  header = FALSE,
  sep = "\t"
)

countmat_6h <- read.delim(
  here("inputs", "countmat", "GM12878_stimulus", "GM_stim_6h_counts_1.tsv"),
  header = FALSE,
  sep = "\t"
)

# read in row metadata
row_meta <- read.delim(
  here("inputs", "bed", "ATAC_peaks", "filtered", "consensus_filt_annot.tsv"),
  sep = "\t"
)

# read in column metadata
col_meta_1h <- read.delim(
  here("inputs", "countmat", "GM12878_stimulus", "metadata.txt"),
  sep = "\t"
)

col_meta_6h <- col_meta_1h

# filter to only include counts (exclude genomic coordinates)
counts_1h <- countmat_1h[, 5:16]
counts_6h <- countmat_6h[, 5:16]

# convert to CPM
cpm_1h <- apply(counts_1h, 2, FUN = function(x) 1000000 * x / sum(x))
cpm_6h <- apply(counts_6h, 2, FUN = function(x) 1000000 * x / sum(x))

# read in differential accessibility results

# stimulus vs vehicle
dar_IFN_1h <- read.delim(
  here("inputs", "DARs", "GM12878_stimulus", "DAR_IFN_1h.tsv"),
  sep = "\t"
)

dar_IFN_6h <- read.delim(
  here("inputs", "DARs", "GM12878_stimulus", "DAR_IFN_6h.tsv"),
  sep = "\t"
)

dar_Dex_1h <- read.delim(
  here("inputs", "DARs", "GM12878_stimulus", "DAR_Dex_1h.tsv"),
  sep = "\t"
)

dar_Dex_6h <- read.delim(
  here("inputs", "DARs", "GM12878_stimulus", "DAR_Dex_6h.tsv"),
  sep = "\t"
)

# BAFinh + stim vs stim
dar_BRM_IFN_1h <- read.delim(
  here("inputs", "DARs", "GM12878_stimulus", "DAR_BRM_IFN_1h.tsv"),
  sep = "\t"
) %>%
  arrange(seqnames, start)

dar_BRM_IFN_6h <- read.delim(
  here("inputs", "DARs", "GM12878_stimulus", "DAR_BRM_IFN_6h.tsv"),
  sep = "\t"
) %>%
  arrange(seqnames, start)

dar_BRM_Dex_1h <- read.delim(
  here("inputs", "DARs", "GM12878_stimulus", "DAR_BRM_Dex_1h.tsv"),
  sep = "\t"
) %>%
  arrange(seqnames, start)

dar_BRM_Dex_6h <- read.delim(
  here("inputs", "DARs", "GM12878_stimulus", "DAR_BRM_Dex_6h.tsv"),
  sep = "\t"
) %>%
  arrange(seqnames, start)

#PROseq
PRO_fc_IFN <- read.delim(
    here("inputs", "countmat", "PROseq_eRNA", "IFN_v_Veh_PROseq_cRE.tsv"),
    sep='\t')
PRO_fc_BAFinh_IFN <- read.delim(
    here("inputs", "countmat", "PROseq_eRNA", "BAFinh_IFN_v_IFN_PROseq_cRE.tsv"),
    sep='\t')
#generate data frame containing only fold changes (BAFinh + stim vs stim) for each stimulus
ifny_fc <- data.frame("t1h"= dar_BRM_IFN_1h$logFC, "t6h"=dar_BRM_IFN_6h$logFC)
dex_fc <- data.frame("t1h"= dar_BRM_Dex_1h$logFC, "t6h"=dar_BRM_Dex_6h$logFC)

# Generate normalized accessibility heatmaps

In [3]:
#generate heatmaps for IFN-inducible cREs (separated by timepoint of induction)
ifn_up_both <- which(dar_IFN_1h$change == "up" & dar_IFN_6h$change == "up")
ifn_up_1h <- which(dar_IFN_1h$change == "up" & dar_IFN_6h$change != "up")
ifn_up_6h <-  which(dar_IFN_1h$change != "up" & dar_IFN_6h$change == "up")
str(ifn_up_both)
str(ifn_up_1h)
str(ifn_up_6h)
n_ifn <- sum(dar_IFN_1h$change == "up" | dar_IFN_6h$change == "up")
#order rows by magnitude of induction (for plotting)
ifn_up_both_ordered <- (dar_IFN_1h[ifn_up_both,] %>% mutate(rownum=ifn_up_both) %>% arrange(desc((logFC*logCPM)-logCPM)))$rownum
ifn_up_1h_ordered <- (dar_IFN_1h[ifn_up_1h,] %>% mutate(rownum=ifn_up_1h) %>% arrange(desc((logFC*logCPM)-logCPM)))$rownum
ifn_up_6h_ordered <- (dar_IFN_6h[ifn_up_6h,] %>% mutate(rownum=ifn_up_6h) %>% arrange(desc((logFC*logCPM)-logCPM)))$rownum

cpm_ifn_all <- cbind(cpm_1h[,1:2],cpm_6h[,1:2], cpm_1h[,3:4],cpm_6h[,3:4], cpm_1h[,9:10], cpm_6h[,9:10]) %>% 
    `colnames<-`(c(rep("Veh_1h",2),rep("Veh_6h",2),rep("IFN_1h",2), rep("IFN_6h",2), rep("BRM_IFN_1h",2), rep("BRM_IFN_6h",2)))

dev.off()
pdf("ifn_1h_hm.pdf", width=2)
hm(cpm_ifn_all,ifn_up_1h_ordered, ifn_col, n=n_ifn, scale="row")
dev.off()

pdf("ifn_both_hm.pdf", width=2)
hm(cpm_ifn_all,ifn_up_both_ordered, ifn_col, n=n_ifn, scale="row")
dev.off()

pdf("ifn_6h_hm.pdf", width=2)
hm(cpm_ifn_all,ifn_up_6h_ordered, ifn_col, n=n_ifn, scale="row")
dev.off()

 int [1:142] 1739 3731 3788 3790 3980 4077 4229 4231 4234 4237 ...
 int [1:406] 76 282 298 387 446 493 557 797 1023 1024 ...
 int [1:1543] 72 322 409 425 713 1022 1183 1388 1671 2169 ...


null device 
          1

pdf 
  3

pdf 
  3

pdf 
  3

In [4]:
dex_up_both <- which(dar_Dex_1h$change == "up" & dar_Dex_6h$change == "up")
dex_up_1h <- which(dar_Dex_1h$change == "up" & dar_Dex_6h$change != "up")
dex_up_6h <-  which(dar_Dex_1h$change != "up" & dar_Dex_6h$change == "up")
str(dex_up_both)
str(dex_up_1h)
str(dex_up_6h)
n_dex <- sum(dar_Dex_1h$change == "up" | dar_Dex_6h$change == "up")

dex_up_both_ordered <- (dar_Dex_1h[dex_up_both,] %>% mutate(rownum=dex_up_both) %>% arrange(desc((logFC*logCPM)-logCPM)))$rownum
dex_up_1h_ordered <- (dar_Dex_1h[dex_up_1h,] %>% mutate(rownum=dex_up_1h) %>% arrange(desc((logFC*logCPM)-logCPM)))$rownum
dex_up_6h_ordered <- (dar_Dex_6h[dex_up_6h,] %>% mutate(rownum=dex_up_6h) %>% arrange(desc((logFC*logCPM)-logCPM)))$rownum

cpm_dex_all <- cbind(cpm_1h[,1:2],cpm_6h[,1:2], cpm_1h[,5:6],cpm_6h[,5:6], cpm_1h[,11:12], cpm_6h[,11:12]) %>% 
    `colnames<-`(c(rep("Veh_1h",2),rep("Veh_6h",2),rep("Dex_1h",2), rep("Dex_6h",2), rep("BRM_Dex_1h",2), rep("BRM_Dex_6h",2)))

dev.off()
pdf("dex_1h_hm.pdf", width=2)
hm(cpm_dex_all,dex_up_1h_ordered, dex_col, n=n_dex, scale="row")
dev.off()
pdf("dex_both_hm.pdf", width=2)
hm(cpm_dex_all,dex_up_both_ordered, dex_col, n=n_dex, scale="row")
dev.off()
pdf("dex_6h_hm.pdf", width=2)
hm(cpm_dex_all,dex_up_6h_ordered, dex_col,n=n_dex, scale="row")
dev.off()

 int [1:1691] 243 378 389 466 580 585 616 622 666 710 ...
 int [1:1406] 12 162 242 586 799 896 1125 1192 1231 1240 ...
 int [1:731] 339 410 415 453 986 1328 1718 1719 2100 2973 ...


pdf 
  3

pdf 
  3

pdf 
  3

pdf 
  3

# Generate log2FC accessibility BAFinh -> stim vs stim alone heatmaps

In [5]:
#fc heatmaps
cols <- c("navy", "white", "firebrick")
pheatmap(ifny_fc[ifn_up_1h_ordered,], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=150/n_ifn, 
        filename="ifn_1h_fc_hm.pdf", width=1, height=2)

pheatmap(ifny_fc[ifn_up_both_ordered,], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=150/n_ifn, 
        filename="ifn_both_fc_hm.pdf", width=1, height=2)

pheatmap(ifny_fc[ifn_up_6h_ordered,], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=150/n_ifn, 
        filename="ifn_6h_fc_hm.pdf", width=1, height=2)

pheatmap(dex_fc[dex_up_1h_ordered,], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=150/n_dex, 
        filename="dex_1h_fc_hm.pdf", width=1, height=2)

pheatmap(dex_fc[dex_up_both_ordered,], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=150/n_dex, 
        filename="dex_both_fc_hm.pdf", width=1, height=2)

pheatmap(dex_fc[dex_up_6h_ordered,], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=150/n_dex, 
        filename="dex_6h_fc_hm.pdf", width=1, height=2)

# Generate log2FC nascent transcription IFN-gamma vs vehicle and  BAFinh -> IFN-gamma vs IFN-gamma alone heatmaps

In [6]:
cols <- c("#006CD1", "#FFFFFF", "#994F00")
pheatmap(PRO_fc_IFN$log2FoldChange[ifn_up_1h_ordered], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=200/n_ifn, 
        filename="PRO_ifn_1h_fc_hm.pdf", cellwidth=20, height=3, main=paste("n =",length(ifn_up_1h_ordered)))

pheatmap(PRO_fc_IFN$log2FoldChange[ifn_up_both_ordered], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=200/n_ifn, 
        filename="PRO_ifn_both_fc_hm.pdf", cellwidth=20, height=3, main=paste("n =",length(ifn_up_both_ordered)))

pheatmap(PRO_fc_IFN$log2FoldChange[ifn_up_6h_ordered], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=200/n_ifn, 
        filename="PRO_ifn_6h_fc_hm.pdf", cellwidth=20, height=3, main=paste("n =",length(ifn_up_6h_ordered)))

pheatmap(PRO_fc_BAFinh_IFN$log2FoldChange[ifn_up_1h_ordered], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=200/n_ifn, 
        filename="PRO_BAFinh_ifn_1h_fc_hm.pdf", cellwidth=20, height=3, main=paste("n =",length(ifn_up_1h_ordered)))

pheatmap(PRO_fc_BAFinh_IFN$log2FoldChange[ifn_up_both_ordered], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=200/n_ifn, 
        filename="PRO_BAFinh_ifn_both_fc_hm.pdf", cellwidth=20, height=3, main=paste("n =",length(ifn_up_both_ordered)))


pheatmap(PRO_fc_BAFinh_IFN$log2FoldChange[ifn_up_6h_ordered], 
         cluster_rows=FALSE, cluster_cols=FALSE,
        color=colorRampPalette(cols)(100), 
         breaks=c(seq(-4,4,length.out=101)),
        show_rownames=FALSE, cellheight=200/n_ifn, 
        filename="PRO_BAFinh_ifn_6h_fc_hm.pdf", cellwidth=20, height=3, main=paste("n =",length(ifn_up_6h_ordered)))